# 1. Contexto e Perguntas de Negócio

Este projeto tem como objetivo construir um pipeline de dados em ambiente de nuvem utilizando o dataset público de e-commerce da Olist.

A partir dos dados de pedidos, produtos, clientes, vendedores, pagamentos e avaliações, o pipeline busca transformar os dados brutos em informações estruturadas para análise.

### Perguntas de negócio

O projeto foi orientado pelas seguintes perguntas:

1. Como o volume de pedidos e o valor das vendas evoluíram ao longo do período analisado?

2. Quais categorias de produtos apresentam maior volume de vendas e maior valor financeiro?

3. Qual é o comportamento dos clientes em relação à recorrência de compras?

4. Como estão distribuídas as avaliações dos clientes?

As perguntas definidas nesta etapa orientam a construção das camadas Silver e Gold e as análises realizadas ao final do projeto.


A primeira etapa do pipeline consiste em disponibilizar os dados de origem no ambiente e realizar sua ingestão na camada Bronze.

Os arquivos originais da Olist foram armazenados em um Volume do Unity Catalog, preservando os dados em seu formato original. A camada Bronze tem como objetivo manter uma representação dos dados de origem sem aplicação de regras de negócio ou transformações analíticas.

Nesta etapa, iniciei pela tabela de pedidos (`orders`), que representa a entidade central do processo de vendas e será posteriormente relacionada às demais tabelas da base.

## 1.1 Leitura do arquivo de pedidos

Antes de criar a tabela Bronze, o arquivo de pedidos é lido diretamente do Volume do Unity Catalog.

Nesta etapa, o objetivo é verificar se o Databricks consegue interpretar corretamente o arquivo CSV, seus cabeçalhos e seus tipos de dados.

In [0]:
%sql
SELECT *
FROM read_files(
    '/Volumes/workspace/default/e_commerce/olist_orders_dataset.csv',
    format => 'csv',
    header => true
);

## 1.2 Criação da tabela Bronze de pedidos

Após validar a leitura do arquivo, os dados são colocados como uma tabela na camada Bronze.

A tabela `bronze_orders` mantém os registros de pedidos em sua estrutura original, mantendo uma fonte estruturada e rastreável para as etapas seguintes.

Neste momento, não são realizadas correções, exclusões ou transformações de negócio.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_orders AS

SELECT *
FROM read_files(
    '/Volumes/workspace/default/e_commerce/olist_orders_dataset.csv',
    format => 'csv',
    header => true
);

## 1.3 Validação da carga

Após a criação da tabela Bronze, é realizada uma validação simples da quantidade de registros carregados.

In [0]:
%sql
SELECT COUNT(*) AS quantidade_registros
FROM workspace.default.bronze_orders;




A tabela `bronze_orders` possui 99.441 registros, confirmando a carga integral dos registros presentes no arquivo de origem.

## 1.4 Inspeção inicial dos registros

Após validar a quantidade de registros, são consultadas algumas linhas da tabela Bronze para verificar visualmente a estrutura dos dados e confirmar que os campos foram interpretados corretamente pelo Databricks.

Essa inspeção também permite observar os valores presentes nas principais colunas da tabela.

In [0]:
%sql
SELECT *
FROM workspace.default.bronze_orders
LIMIT 10;

## 1.5 Identificação do schema

Nesta etapa, é realizada a identificação das colunas e dos respectivos tipos de dados da tabela Bronze.

In [0]:
%sql
DESCRIBE workspace.default.bronze_orders;

## 1.6 Distribuição dos pedidos por status

Para compreender o comportamento da base, é analisada a distribuição dos pedidos.

Essa análise permite identificar a composição da base.

In [0]:
%sql
SELECT
    order_status,
    COUNT(*) AS quantidade_pedidos
FROM workspace.default.bronze_orders
GROUP BY order_status
ORDER BY quantidade_pedidos DESC;


96% dos pedidos apresenta status `delivered`. Porém, também existem pedidos em diferentes etapas do processo, como `shipped`, `canceled`, `unavailable`, `invoiced`, `processing`, `created` e `approved`.

A existência de diferentes status indica que as datas de entrega não devem ser avaliadas isoladamente, pois a ausência de determinada data pode ser esperada dependendo da situação do pedido.

## 1.7 Análise de completude das datas

A primeira verificação de qualidade consiste em avaliar a completude dos principais campos de data do ciclo do pedido.

In [0]:
%sql
SELECT
    COUNT(*) AS total_pedidos,
    COUNT(order_approved_at) AS pedidos_com_aprovacao,
    COUNT(order_delivered_carrier_date) AS pedidos_com_envio,
    COUNT(order_delivered_customer_date) AS pedidos_entregues,
    COUNT(order_estimated_delivery_date) AS pedidos_com_previsao
FROM workspace.default.bronze_orders;

### Resultado

A maior parte dos registros apresenta as principais datas preenchidas. A data de previsão de entrega está presente em todos os pedidos, enquanto existem registros sem data de aprovação, envio à transportadora ou entrega ao cliente.

A ausência de uma data não é considerada automaticamente um erro, pois sua validade depende do status do pedido.

## 1.8 Consistência entre status e data de entrega

Para avaliar se os valores ausentes são esperados, os registros são analisados considerando simultaneamente o status do pedido e a existência da data de entrega ao cliente.

Essa abordagem permite diferenciar uma ausência de informação potencialmente legítima de uma possível inconsistência de dados.

In [0]:
%sql
SELECT
    order_status,
    COUNT(*) AS total_pedidos,
    SUM(
        CASE
            WHEN order_delivered_customer_date IS NULL THEN 1
            ELSE 0
        END
    ) AS sem_data_entrega
FROM workspace.default.bronze_orders
GROUP BY order_status
ORDER BY total_pedidos DESC;

### Resultado

A análise identificou que a ausência da data de entrega está concentrada principalmente em pedidos que não apresentam status `delivered`, o que pode ser esperado para pedidos ainda em processamento ou que não foram concluídos.

Também foram identificados registros que merecem investigação adicional por apresentarem combinação potencialmente inconsistente entre o status do pedido e a data de entrega.

In [0]:
%sql
SELECT
    order_id,
    order_status,
    order_delivered_customer_date
FROM workspace.default.bronze_orders
WHERE
    (order_status = 'delivered'
     AND order_delivered_customer_date IS NULL)
    OR
    (order_status <> 'delivered'
     AND order_delivered_customer_date IS NOT NULL);

## 1.9 Investigação de inconsistências

Com base na análise anterior, foram selecionados os registros que apresentam uma das seguintes situações:

- pedido com status `delivered` e sem data de entrega ao cliente;
- pedido com status diferente de `delivered` e com data de entrega ao cliente preenchida.

In [0]:
%sql
SELECT
    order_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM workspace.default.bronze_orders
WHERE
    (order_status = 'delivered'
     AND order_delivered_customer_date IS NULL)
    OR
    (order_status <> 'delivered'
     AND order_delivered_customer_date IS NOT NULL)
ORDER BY order_status, order_purchase_timestamp;

### Regra de qualidade identificada

Foram identificados 14 registros que apresentam inconsistências aparentes entre o status do pedido e a data de entrega:

- 8 pedidos com status `delivered` sem data de entrega ao cliente;
- 6 pedidos com status `canceled` com data de entrega ao cliente preenchida.

## 2. Ingestão dos itens dos pedidos

A tabela `order_items` contém os itens associados aos pedidos, incluindo informações de produto, vendedor, preço e frete.

Essa tabela possui uma granularidade diferente de `orders`: enquanto cada registro de `orders` representa um pedido, cada registro de `order_items` representa um item dentro de um pedido.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_order_items AS

SELECT *
FROM read_files(
    '/Volumes/workspace/default/e_commerce/olist_order_items_dataset.csv',
    format => 'csv',
    header => true
);

## 2.1 Validação da carga de itens

Após a criação da tabela Bronze, é verificada a quantidade de registros carregados para garantir a correspondência com o arquivo de origem.

In [0]:
%sql
SELECT COUNT(*) AS quantidade_registros
FROM workspace.default.bronze_order_items;

### Resultado

A tabela `bronze_order_items` possui 112.650 registros, correspondentes à quantidade de registros existente no arquivo de origem.

## 2.2 Identificação do schema

O schema da tabela `order_items` é analisado para identificar os campos disponíveis, seus tipos de dados e os principais atributos utilizados para relacionar pedidos, produtos e vendedores.

In [0]:
%sql
DESCRIBE workspace.default.bronze_order_items;

## 2.3 Inspeção dos registros
Consultei alguns registros da tabela para validar visualmente os dados carregados e compreender a estrutura de cada item de pedido.

In [0]:
%sql
SELECT *
FROM workspace.default.bronze_order_items
LIMIT 10;

## 2.4 Análise da granularidade e dos relacionamentos

Nesta etapa, são analisadas as quantidades de linhas, pedidos, produtos e vendedores distintos.

O objetivo é compreender a granularidade da tabela e identificar como ela se relaciona com as demais entidades do modelo.

A ideia é verificar a unicidade da combinação `order_id` + `order_item_id`, que representa uma possível chave composta para a tabela.

In [0]:
%sql

SELECT
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    COUNT(DISTINCT product_id) AS produtos_distintos,
    COUNT(DISTINCT seller_id) AS vendedores_distintos
FROM workspace.default.bronze_order_items;

### Resultado

A tabela possui 112.650 registros e 98.666 pedidos distintos, demonstrando que um pedido pode possuir múltiplos itens.

A combinação `order_id` + `order_item_id` não apresentou duplicidades na verificação realizada, sendo considerada uma chave candidata composta para identificar unicamente os itens dos pedidos.

A tabela também possui referências a 32.951 produtos e 3.095 vendedores distintos.

In [0]:
%sql

SELECT
    order_id,
    order_item_id,
    COUNT(*) AS quantidade
FROM workspace.default.bronze_order_items
GROUP BY order_id, order_item_id
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

## 3. Ingestão de Produtos
A tabela de produtos contém informações cadastrais dos produtos comercializados na plataforma.

Nesta etapa, o arquivo original é carregado para a camada Bronze sem aplicação de regras de negócio ou transformações. O objetivo é preservar uma representação estruturada dos dados de origem para utilização nas etapas posteriores do pipeline.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_products AS

SELECT *
FROM read_files(
    '/Volumes/workspace/default/e_commerce/olist_products_dataset.csv',
    format => 'csv',
    header => true
);

In [0]:
%sql
SELECT COUNT(*) AS quantidade_registros
FROM workspace.default.bronze_products;

In [0]:
%sql
DESCRIBE workspace.default.bronze_products;

## 4. Ingestão de Clientes — Camada Bronze

A tabela de clientes contém informações relacionadas aos consumidores dos pedidos, incluindo seu identificador e localização geográfica.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_customers AS

SELECT *
FROM read_files(
    '/Volumes/workspace/default/e_commerce/olist_customers_dataset.csv',
    format => 'csv',
    header => true
);

In [0]:
%sql
SELECT COUNT(*) AS quantidade_registros
FROM workspace.default.bronze_customers;

In [0]:
%sql
DESCRIBE workspace.default.bronze_customers;

## 4.1 Análise da granularidade e identificação dos clientes

A tabela de clientes possui dois identificadores distintos: `customer_id` e `customer_unique_id`.

Nesta etapa, será avaliada a quantidade de identificadores distintos para compreender a granularidade da tabela e verificar se um mesmo cliente pode estar associado a mais de um `customer_id`.

Essa análise é importante para evitar interpretações incorretas nas análises posteriores de clientes e pedidos.

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT customer_id) AS customer_ids_distintos,
    COUNT(DISTINCT customer_unique_id) AS customer_unique_ids_distintos
FROM workspace.default.bronze_customers;

In [0]:
%sql

SELECT
    customer_unique_id,
    COUNT(DISTINCT customer_id) AS quantidade_customer_ids
FROM workspace.default.bronze_customers
GROUP BY customer_unique_id
HAVING COUNT(DISTINCT customer_id) > 1
ORDER BY quantidade_customer_ids DESC;

### Resultado da análise

A tabela possui 99.441 registros e 99.441 `customer_id` distintos, indicando que cada registro possui um identificador de cliente associado.

Entretanto, foram identificados 96.096 `customer_unique_id` distintos. A existência de menos clientes únicos do que registros indica que um mesmo cliente pode estar associado a diferentes `customer_id`.

Essa distinção será considerada nas análises posteriores: `customer_id` será utilizado para relacionamentos com os pedidos, enquanto `customer_unique_id` será utilizado quando o objetivo for analisar clientes de forma única.

## 5. Ingestão de Vendedores

A tabela de vendedores contém informações cadastrais dos vendedores que participam das transações da plataforma.

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.bronze_sellers AS

SELECT *
FROM read_files(
    '/Volumes/workspace/default/e_commerce/olist_sellers_dataset.csv',
    format => 'csv',
    header => true
);

## 5.1 Validação da carga

Tabela validada.

In [0]:
%sql

SELECT COUNT(*) AS quantidade_registros
FROM workspace.default.bronze_sellers;

In [0]:
%sql

DESCRIBE workspace.default.bronze_sellers;

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT seller_id) AS vendedores_distintos
FROM workspace.default.bronze_sellers;

In [0]:
%sql

SELECT
    seller_id,
    COUNT(*) AS quantidade
FROM workspace.default.bronze_sellers
GROUP BY seller_id
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

### Resultado

A análise confirma a granularidade cadastral da tabela de vendedores. O identificador `seller_id` não apresentou duplicidades na verificação realizada e pode ser utilizado como chave para relacionar os vendedores aos itens dos pedidos.

## 6. Ingestão de Pagamentos — Camada Bronze

A tabela de pagamentos contém informações sobre os pagamentos associados aos pedidos, incluindo método de pagamento, número de parcelas e valor.

Assim como nas demais fontes, os dados são inicialmente carregados para a camada Bronze sem aplicação de transformações ou regras de negócio.

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.bronze_order_payments AS

SELECT *
FROM read_files(
    '/Volumes/workspace/default/e_commerce/olist_order_payments_dataset.csv',
    format => 'csv',
    header => true
);

In [0]:
%sql

SELECT COUNT(*) AS quantidade_registros
FROM workspace.default.bronze_order_payments;

## Resultado
A tabela possui 103.886 registros.

In [0]:
%sql

DESCRIBE workspace.default.bronze_order_payments;

In [0]:
%sql

SELECT
    COUNT(*) AS total_pagamentos,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    COUNT(DISTINCT payment_type) AS tipos_pagamento_distintos
FROM workspace.default.bronze_order_payments;

In [0]:
%sql

SELECT
    order_id,
    COUNT(*) AS quantidade_pagamentos
FROM workspace.default.bronze_order_payments
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY quantidade_pagamentos DESC;

### Resultado

A tabela de pagamentos possui 103.886 registros associados a 99.440 pedidos distintos. A existência de múltiplos registros de pagamento para um mesmo pedido confirma uma relação de um para muitos entre pedidos e pagamentos.

## 7. Ingestão de Avaliações

A tabela de avaliações contém informações fornecidas pelos clientes sobre os pedidos, incluindo nota, título, comentário e datas relacionadas à avaliação.

In [0]:
%sql


CREATE OR REPLACE TABLE workspace.default.bronze_order_reviews AS

SELECT *
FROM read_files(
    '/Volumes/workspace/default/e_commerce/olist_order_reviews_dataset.csv',
    format => 'csv',
    header => true,
    multiLine => true
);

In [0]:
%sql

SELECT COUNT(*) AS quantidade_registros
FROM workspace.default.bronze_order_reviews;

In [0]:
%sql

DESCRIBE workspace.default.bronze_order_reviews;

In [0]:
%sql

SELECT
    COUNT(*) AS total_avaliacoes,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    COUNT(DISTINCT review_id) AS reviews_distintos
FROM workspace.default.bronze_order_reviews;

In [0]:
%sql

SELECT
    order_id,
    COUNT(*) AS quantidade_reviews
FROM workspace.default.bronze_order_reviews
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY quantidade_reviews DESC;

In [0]:
%sql

SELECT *
FROM workspace.default.bronze_order_reviews
WHERE order_id IS NULL
LIMIT 20;

In [0]:
%sql

SELECT
    order_id,
    review_id,
    review_score,
    review_creation_date,
    review_answer_timestamp
FROM workspace.default.bronze_order_reviews
WHERE order_id IS NOT NULL
  AND order_id LIKE '201%'
LIMIT 20;

In [0]:
%sql

SELECT
    review_score,
    COUNT(*) AS quantidade
FROM workspace.default.bronze_order_reviews
GROUP BY review_score
ORDER BY review_score;

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.bronze_order_reviews AS

SELECT *
FROM read_files(
    '/Volumes/workspace/default/e_commerce/olist_order_reviews_dataset.csv',
    format => 'csv',
    header => true,
    multiLine => true,
    schema => '
        review_id string,
        order_id string,
        review_score int,
        review_comment_title string,
        review_comment_message string,
        review_creation_date timestamp,
        review_answer_timestamp timestamp
    '
);

In [0]:
%sql

SELECT COUNT(*) AS quantidade_registros
FROM workspace.default.bronze_order_reviews;

In [0]:
%sql

DESCRIBE workspace.default.bronze_order_reviews;

In [0]:
%sql

SELECT
    review_score,
    COUNT(*) AS quantidade
FROM workspace.default.bronze_order_reviews
GROUP BY review_score
ORDER BY review_score;

### Resultado da ingestão

A tabela de avaliações foi carregada na camada Bronze com schema explicitamente definido para garantir a correta interpretação dos campos numéricos e de data.

A carga resultou em 99.249 registros. O campo `review_score` foi interpretado como inteiro, apresentando valores entre 1 e 5. Foram identificados 26 registros sem avaliação (`NULL`), que serão considerados posteriormente na etapa de qualidade e tratamento dos dados.

A estrutura da tabela apresenta sete atributos: identificador da avaliação, identificador do pedido, nota, título e comentário da avaliação e as respectivas datas de criação e resposta.

## 8. Transformação dos Dados — Camada Silver

A camada Silver tem como objetivo transformar os dados carregados na camada Bronze em estruturas mais consistentes e adequadas para análise.

Nesta etapa serão realizadas validações, padronizações de tipos, tratamento de valores nulos e criação de atributos derivados quando necessário.

As tabelas Silver manterão a granularidade original dos dados sempre que possível, preservando os identificadores necessários para os relacionamentos entre as entidades.

### 8.1 Tratamento da tabela de pedidos

A tabela de pedidos representa a entidade central do processo de compra. Cada registro corresponde a um pedido realizado na plataforma.

Nesta etapa serão mantidos os principais atributos do pedido, com padronização dos campos de data e criação de atributos derivados para facilitar análises temporais.

In [0]:
%sql

SELECT
    COUNT(*) AS total_pedidos,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    COUNT(*) - COUNT(order_id) AS order_id_nulos,
    COUNT(*) - COUNT(order_status) AS status_nulos,
    COUNT(*) - COUNT(order_purchase_timestamp) AS data_compra_nulos,
    COUNT(*) - COUNT(order_estimated_delivery_date) AS data_estimada_nulos
FROM workspace.default.bronze_orders;

### 8.2 Validação dos status dos pedidos

Antes da transformação, foi realizada uma validação dos valores presentes em `order_status`, com o objetivo de verificar a consistência da classificação dos pedidos e identificar possíveis valores inesperados.

In [0]:
%sql

SELECT
    order_status,
    COUNT(*) AS quantidade
FROM workspace.default.bronze_orders
GROUP BY order_status
ORDER BY quantidade DESC;

### 8.3 Criação da tabela Silver de pedidos

A partir da tabela Bronze, foi criada uma versão tratada dos pedidos.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_orders AS

SELECT
    order_id,
    customer_id,
    order_status,
    
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date,

    YEAR(order_purchase_timestamp) AS ano_compra,
    MONTH(order_purchase_timestamp) AS mes_compra

FROM workspace.default.bronze_orders
WHERE order_id IS NOT NULL;

### 8.4 Validação da tabela Silver

Após a transformação, foram realizadas verificações para confirmar a quantidade de registros, a unicidade dos pedidos e a presença dos principais atributos utilizados no relacionamento com as demais tabelas.

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    COUNT(*) - COUNT(order_id) AS order_id_nulos,
    COUNT(*) - COUNT(customer_id) AS customer_id_nulos
FROM workspace.default.silver_orders;

In [0]:
%sql

SELECT
    order_id,
    COUNT(*) AS quantidade
FROM workspace.default.silver_orders
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

Um registro = Um pedido

In [0]:
%sql

SELECT *
FROM workspace.default.silver_orders
ORDER BY order_purchase_timestamp
LIMIT 20;

### Resultado da validação

A tabela `silver_orders` manteve os 99.441 pedidos presentes na camada Bronze. Não foram identificados valores nulos nos campos `order_id` e `customer_id`, e a quantidade de pedidos distintos permaneceu igual à quantidade total de registros.

A verificação de duplicidade também não retornou registros, confirmando a granularidade de um registro por pedido na tabela Silver.

### 8.5 Tratamento da tabela de itens dos pedidos

A tabela de itens representa os produtos associados a cada pedido. Diferentemente da tabela de pedidos, sua granularidade é definida por item, permitindo que um mesmo pedido possua múltiplos registros.

Nesta etapa serão validados os identificadores dos pedidos, produtos e vendedores, além dos valores de preço e frete. A granularidade de um registro por item será preservada para permitir análises detalhadas posteriormente.

In [0]:
%sql

SELECT
    COUNT(*) AS total_itens,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    COUNT(DISTINCT product_id) AS produtos_distintos,
    COUNT(DISTINCT seller_id) AS vendedores_distintos,
    COUNT(*) - COUNT(order_id) AS order_id_nulos,
    COUNT(*) - COUNT(product_id) AS product_id_nulos,
    COUNT(*) - COUNT(seller_id) AS seller_id_nulos,
    COUNT(*) - COUNT(price) AS price_nulos,
    COUNT(*) - COUNT(freight_value) AS frete_nulos
FROM workspace.default.bronze_order_items;

### 8.6 Validação da granularidade dos itens

Como um pedido pode possuir múltiplos itens, a granularidade da tabela é definida pela combinação entre `order_id` e `order_item_id`.



In [0]:
%sql

SELECT
    order_id,
    order_item_id,
    COUNT(*) AS quantidade
FROM workspace.default.bronze_order_items
GROUP BY
    order_id,
    order_item_id
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

### 8.7 Validação dos valores financeiros

Também foram avaliados os campos `price` e `freight_value`, verificando a existência de valores negativos que poderiam indicar inconsistências nos dados financeiros dos itens.

In [0]:
%sql

SELECT
    COUNT(*) AS total_itens,
    SUM(CASE WHEN price < 0 THEN 1 ELSE 0 END) AS precos_negativos,
    SUM(CASE WHEN freight_value < 0 THEN 1 ELSE 0 END) AS fretes_negativos,
    MIN(price) AS menor_preco,
    MAX(price) AS maior_preco,
    MIN(freight_value) AS menor_frete,
    MAX(freight_value) AS maior_frete
FROM workspace.default.bronze_order_items;

### Resultado da validação

A tabela de itens possui 112.650 registros e não apresenta valores nulos nos principais identificadores ou nos campos financeiros avaliados.

Também não foram identificados preços ou valores de frete negativos. Os valores observados variam de R$ 0,85 a R$ 6.735,00 para os produtos e de R$ 0,00 a R$ 409,68 para o frete.

A verificação da combinação `order_id` e `order_item_id` não identificou duplicidades, confirmando a granularidade de um registro por item do pedido.

### 8.8 Criação da tabela Silver de itens dos pedidos


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.silver_order_items AS

SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value,
    price + freight_value AS valor_total_item

FROM workspace.default.bronze_order_items
WHERE order_id IS NOT NULL
  AND product_id IS NOT NULL
  AND seller_id IS NOT NULL;

### 8.9 Validação da tabela Silver de itens

Após a transformação, foram realizadas verificações para confirmar a preservação da quantidade de registros, a unicidade da combinação `order_id` e `order_item_id` e a consistência do novo campo de valor total do item.

In [0]:
%sql

SELECT
    COUNT(*) AS total_itens,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    COUNT(DISTINCT product_id) AS produtos_distintos,
    COUNT(DISTINCT seller_id) AS vendedores_distintos,
    COUNT(*) - COUNT(valor_total_item) AS valor_total_nulo
FROM workspace.default.silver_order_items;

In [0]:
%sql

SELECT
    order_id,
    order_item_id,
    COUNT(*) AS quantidade
FROM workspace.default.silver_order_items
GROUP BY
    order_id,
    order_item_id
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

In [0]:
%sql

SELECT
    order_id,
    order_item_id,
    price,
    freight_value,
    valor_total_item
FROM workspace.default.silver_order_items
LIMIT 20;

### 8.10 Tratamento da tabela de produtos

In [0]:
%sql

SELECT
    COUNT(*) AS total_produtos,
    COUNT(DISTINCT product_id) AS produtos_distintos,
    COUNT(*) - COUNT(product_id) AS product_id_nulos,
    COUNT(*) - COUNT(product_category_name) AS categoria_nulos,
    COUNT(*) - COUNT(product_weight_g) AS peso_nulos,
    COUNT(*) - COUNT(product_length_cm) AS comprimento_nulos,
    COUNT(*) - COUNT(product_height_cm) AS altura_nulos,
    COUNT(*) - COUNT(product_width_cm) AS largura_nulos,
    COUNT(*) - COUNT(product_photos_qty) AS fotos_nulos
FROM workspace.default.bronze_products;

### 8.11 Validação dos atributos físicos

Os atributos físicos dos produtos foram avaliados para identificar valores negativos ou inconsistentes que poderiam comprometer análises relacionadas às características dos produtos.

In [0]:
%sql

SELECT
    SUM(CASE WHEN product_weight_g < 0 THEN 1 ELSE 0 END) AS pesos_negativos,
    SUM(CASE WHEN product_length_cm < 0 THEN 1 ELSE 0 END) AS comprimentos_negativos,
    SUM(CASE WHEN product_height_cm < 0 THEN 1 ELSE 0 END) AS alturas_negativas,
    SUM(CASE WHEN product_width_cm < 0 THEN 1 ELSE 0 END) AS larguras_negativas,
    SUM(CASE WHEN product_photos_qty < 0 THEN 1 ELSE 0 END) AS fotos_negativas
FROM workspace.default.bronze_products;

### Resultado da validação

A tabela de produtos possui 32.951 registros e 32.951 identificadores distintos, não sendo identificados valores nulos em `product_id`.

Foram identificados 610 produtos sem categoria e apenas dois registros sem informação de peso e dimensões. Esses valores nulos serão preservados na camada Silver, evitando a atribuição de informações não presentes na fonte original.

Também não foram identificados valores negativos nos atributos físicos avaliados.

### 8.12 Criação da tabela Silver de produtos


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.silver_products AS

SELECT
    product_id,
    product_category_name,
    product_name_lenght,
    product_description_lenght,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm

FROM workspace.default.bronze_products
WHERE product_id IS NOT NULL;

### 8.13 Validação da tabela Silver de produtos

Após a criação da tabela Silver, foi realizada uma validação para confirmar a preservação da quantidade de produtos, a unicidade do identificador e a manutenção dos principais atributos da entidade.

In [0]:
%sql

SELECT
    COUNT(*) AS total_produtos,
    COUNT(DISTINCT product_id) AS produtos_distintos,
    COUNT(*) - COUNT(product_id) AS product_id_nulos,
    COUNT(*) - COUNT(product_category_name) AS categoria_nulos,
    COUNT(*) - COUNT(product_weight_g) AS peso_nulos
FROM workspace.default.silver_products;

In [0]:
%sql

SELECT
    product_id,
    COUNT(*) AS quantidade
FROM workspace.default.silver_products
GROUP BY product_id
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

### 8.14 Tratamento da tabela de clientes

A tabela de clientes contém os identificadores dos clientes e suas informações geográficas.

Nesta etapa serão avaliados os identificadores e os atributos de localização, com atenção especial à diferença entre `customer_id` e `customer_unique_id`. Essa distinção é importante porque um mesmo cliente pode estar associado a diferentes registros de pedidos.

In [0]:
%sql

SELECT
    COUNT(*) AS total_clientes,
    COUNT(DISTINCT customer_id) AS customer_ids_distintos,
    COUNT(DISTINCT customer_unique_id) AS customer_unique_ids_distintos,
    COUNT(*) - COUNT(customer_id) AS customer_id_nulos,
    COUNT(*) - COUNT(customer_unique_id) AS customer_unique_id_nulos,
    COUNT(*) - COUNT(customer_city) AS cidade_nulos,
    COUNT(*) - COUNT(customer_state) AS estado_nulos
FROM workspace.default.bronze_customers;

### 8.15 Análise da granularidade dos clientes

Foi analisada a relação entre `customer_id` e `customer_unique_id` para verificar se um mesmo cliente pode estar associado a mais de um registro de pedido.

Essa análise permite diferenciar o identificador do registro do pedido do identificador do cliente ao longo de sua jornada de compras.

In [0]:
%sql

SELECT
    customer_unique_id,
    COUNT(DISTINCT customer_id) AS quantidade_customer_ids
FROM workspace.default.bronze_customers
GROUP BY customer_unique_id
HAVING COUNT(DISTINCT customer_id) > 1
ORDER BY quantidade_customer_ids DESC
LIMIT 20;

### 8.16 Validação dos atributos geográficos



In [0]:
%sql

SELECT
    COUNT(DISTINCT customer_state) AS estados_distintos,
    COUNT(DISTINCT customer_city) AS cidades_distintas,
    COUNT(*) - COUNT(customer_city) AS cidades_nulas,
    COUNT(*) - COUNT(customer_state) AS estados_nulos
FROM workspace.default.bronze_customers;

In [0]:
%sql

SELECT
    customer_state,
    COUNT(*) AS quantidade
FROM workspace.default.bronze_customers
GROUP BY customer_state
ORDER BY quantidade DESC;

### 8.17 Criação da tabela Silver de clientes

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.silver_customers AS

SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state

FROM workspace.default.bronze_customers
WHERE customer_id IS NOT NULL
  AND customer_unique_id IS NOT NULL;

### 8.18 Validação da tabela Silver de clientes

Após a transformação, foram realizadas verificações para confirmar a preservação dos registros, a unicidade de `customer_id` e a disponibilidade do identificador `customer_unique_id` para análises de recorrência.

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT customer_id) AS customer_ids_distintos,
    COUNT(DISTINCT customer_unique_id) AS customer_unique_ids_distintos,
    COUNT(*) - COUNT(customer_id) AS customer_id_nulos,
    COUNT(*) - COUNT(customer_unique_id) AS customer_unique_id_nulos
FROM workspace.default.silver_customers;

In [0]:
%sql

SELECT
    customer_id,
    COUNT(*) AS quantidade
FROM workspace.default.silver_customers
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

### Resultado da validação

A tabela `silver_customers` manteve os 99.441 registros da camada Bronze e apresentou 99.441 `customer_id` distintos, sem valores nulos ou duplicidades nesse identificador.

Foram identificados 96.096 `customer_unique_id` distintos. A diferença entre os dois identificadores representa a existência de clientes associados a mais de um registro de pedido, característica importante para análises de recorrência e comportamento de compra.

### 8.19 Tratamento da tabela de vendedores

In [0]:
%sql

SELECT
    COUNT(*) AS total_vendedores,
    COUNT(DISTINCT seller_id) AS vendedores_distintos,
    COUNT(*) - COUNT(seller_id) AS seller_id_nulos,
    COUNT(*) - COUNT(seller_city) AS cidade_nulos,
    COUNT(*) - COUNT(seller_state) AS estado_nulos
FROM workspace.default.bronze_sellers;

### 8.20 Criação da tabela Silver de vendedores

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.silver_sellers AS

SELECT
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state

FROM workspace.default.bronze_sellers
WHERE seller_id IS NOT NULL;

In [0]:
%sql

SELECT
    COUNT(*) AS total_vendedores,
    COUNT(DISTINCT seller_id) AS vendedores_distintos,
    COUNT(*) - COUNT(seller_id) AS seller_id_nulos
FROM workspace.default.silver_sellers;

In [0]:
%sql

SELECT
    seller_id,
    COUNT(*) AS quantidade
FROM workspace.default.silver_sellers
GROUP BY seller_id
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

### 8.21 Tratamento da tabela de pagamentos

In [0]:
%sql

SELECT
    COUNT(*) AS total_pagamentos,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    COUNT(DISTINCT payment_sequential) AS sequencias_distintas,
    COUNT(*) - COUNT(order_id) AS order_id_nulos,
    COUNT(*) - COUNT(payment_type) AS tipo_pagamento_nulos,
    COUNT(*) - COUNT(payment_installments) AS parcelas_nulas,
    COUNT(*) - COUNT(payment_value) AS valor_pagamento_nulos
FROM workspace.default.bronze_order_payments;

### 8.23 Validação dos tipos de pagamento

Foi realizada uma análise dos tipos de pagamento presentes na base para identificar as categorias disponíveis e verificar possíveis valores inesperados.

In [0]:
%sql

SELECT
    payment_type,
    COUNT(*) AS quantidade
FROM workspace.default.bronze_order_payments
GROUP BY payment_type
ORDER BY quantidade DESC;

### 8.24 Validação dos valores de pagamento

Os valores de pagamento e o número de parcelas foram avaliados para identificar valores negativos ou inconsistentes que poderiam comprometer as análises financeiras posteriores.

In [0]:
%sql

SELECT
    SUM(CASE WHEN payment_value < 0 THEN 1 ELSE 0 END) AS pagamentos_negativos,
    SUM(CASE WHEN payment_installments < 0 THEN 1 ELSE 0 END) AS parcelas_negativas,
    MIN(payment_value) AS menor_pagamento,
    MAX(payment_value) AS maior_pagamento,
    MIN(payment_installments) AS menor_quantidade_parcelas,
    MAX(payment_installments) AS maior_quantidade_parcelas
FROM workspace.default.bronze_order_payments;

### 8.25 Validação da granularidade dos pagamentos

In [0]:
%sql

SELECT
    order_id,
    payment_sequential,
    COUNT(*) AS quantidade
FROM workspace.default.bronze_order_payments
GROUP BY
    order_id,
    payment_sequential
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

### 8.26 Criação da tabela Silver de pagamentos


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.silver_order_payments AS

SELECT
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value

FROM workspace.default.bronze_order_payments
WHERE order_id IS NOT NULL;

### 8.27 Validação da tabela Silver de pagamentos

In [0]:
%sql

SELECT
    COUNT(*) AS total_pagamentos,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    COUNT(DISTINCT payment_type) AS tipos_pagamento,
    COUNT(*) - COUNT(order_id) AS order_id_nulos,
    COUNT(*) - COUNT(payment_value) AS valor_pagamento_nulos
FROM workspace.default.silver_order_payments;

## 8.28 - Confirmandoa  granularidade

In [0]:
%sql

SELECT
    order_id,
    payment_sequential,
    COUNT(*) AS quantidade
FROM workspace.default.silver_order_payments
GROUP BY
    order_id,
    payment_sequential
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

## 8.29 - Inspeção dos pagamentos

In [0]:
%sql

SELECT
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
FROM workspace.default.silver_order_payments
ORDER BY payment_value DESC
LIMIT 20;

### 8.30 Tratamento da tabela de avaliações


In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.silver_order_reviews AS

SELECT
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp

FROM workspace.default.bronze_order_reviews
WHERE order_id IS NOT NULL;

### 8.31 Validação da tabela Silver de avaliações


In [0]:
%sql

SELECT
    COUNT(*) AS total_avaliacoes,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    COUNT(DISTINCT review_id) AS reviews_distintos,
    COUNT(*) - COUNT(review_score) AS notas_nulas,
    COUNT(*) - COUNT(order_id) AS order_id_nulos
FROM workspace.default.silver_order_reviews;

## 8.32 Validação das notas

In [0]:
%sql

SELECT
    review_score,
    COUNT(*) AS quantidade
FROM workspace.default.silver_order_reviews
GROUP BY review_score
ORDER BY review_score;

## 8.33 - Confirmar os registros sem pedido

In [0]:
%sql
SELECT
    COUNT(*) AS registros_sem_pedido
FROM workspace.default.silver_order_reviews
WHERE order_id IS NULL;

### Resultado da validação

A tabela Silver de avaliações passou a conter apenas registros associados a pedidos identificados, permitindo seu relacionamento com as demais entidades do modelo.

Os registros sem `order_id` foram excluídos da Silver, mas permanecem preservados na camada Bronze. Os 26 registros sem `review_score` foram mantidos para que a ausência da informação seja considerada na etapa de qualidade dos dados.

A distribuição de `review_score` permanece restrita à escala de 1 a 5, além dos valores nulos identificados.

## 9. Modelagem da Camada Gold

A camada Gold tem como objetivo disponibilizar dados consolidados e orientados às perguntas de negócio definidas para o MVP.

Diferentemente da camada Silver, que mantém as entidades tratadas em suas respectivas granularidades, a camada Gold utilizará agregações e relacionamentos entre as entidades para facilitar a análise dos pedidos, vendas, clientes e produtos.

Foram definidas três estruturas analíticas principais:

- `gold_orders`: visão consolidada dos pedidos e seus principais indicadores;
- `gold_product_sales`: visão de vendas por produto e categoria;
- `gold_customer_sales`: visão consolidada do comportamento de compra dos clientes.

Essas estruturas serão utilizadas posteriormente para responder às perguntas de negócio do projeto.

### 9.1 Granularidade da tabela de pedidos

A tabela `gold_orders` terá como granularidade um registro por pedido.

Como as tabelas de itens e pagamentos possuem relações de um para muitos com os pedidos, seus valores serão previamente agregados por `order_id` antes da realização da junção com a tabela de pedidos.

Essa abordagem evita a multiplicação de registros causada pela combinação direta de duas relações de um para muitos.

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_order_items_summary AS

SELECT
    order_id,
    COUNT(*) AS quantidade_itens,
    SUM(price) AS valor_produtos,
    SUM(freight_value) AS valor_frete,
    SUM(valor_total_item) AS valor_total_itens

FROM workspace.default.silver_order_items

GROUP BY order_id;

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_order_payments_summary AS

SELECT
    order_id,
    COUNT(*) AS quantidade_pagamentos,
    SUM(payment_value) AS valor_total_pago

FROM workspace.default.silver_order_payments

GROUP BY order_id;

### 9.2 Criação da visão consolidada dos pedidos

Após a agregação dos itens e pagamentos por pedido, as informações foram integradas à tabela de pedidos.

A estrutura resultante apresenta um registro por pedido e consolida indicadores de quantidade de itens, valores de produtos, frete e pagamentos.

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_orders AS

SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    o.ano_compra,
    o.mes_compra,

    COALESCE(i.quantidade_itens, 0) AS quantidade_itens,
    COALESCE(i.valor_produtos, 0) AS valor_produtos,
    COALESCE(i.valor_frete, 0) AS valor_frete,
    COALESCE(i.valor_total_itens, 0) AS valor_total_itens,

    COALESCE(p.quantidade_pagamentos, 0) AS quantidade_pagamentos,
    COALESCE(p.valor_total_pago, 0) AS valor_total_pago

FROM workspace.default.silver_orders o

LEFT JOIN workspace.default.gold_order_items_summary i
    ON o.order_id = i.order_id

LEFT JOIN workspace.default.gold_order_payments_summary p
    ON o.order_id = p.order_id;

### 9.3 Validação da tabela Gold de pedidos

A validação busca confirmar que a tabela Gold mantém a granularidade de um registro por pedido e que as agregações financeiras não provocaram duplicidades.

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT order_id) AS pedidos_distintos,
    COUNT(*) - COUNT(order_id) AS order_id_nulos
FROM workspace.default.gold_orders;

In [0]:
%sql

SELECT
    order_id,
    COUNT(*) AS quantidade
FROM workspace.default.gold_orders
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

In [0]:
%sql

SELECT
    order_id,
    valor_produtos,
    valor_frete,
    valor_total_itens,
    valor_total_pago
FROM workspace.default.gold_orders
WHERE ABS(valor_total_itens - (valor_produtos + valor_frete)) > 0.01
LIMIT 20;

### 9.4 Visão de vendas por produto

A tabela `gold_product_sales` consolida os itens vendidos por produto, permitindo analisar o volume de vendas e os valores movimentados por produto e categoria.

A granularidade será de um registro por produto, mantendo o relacionamento com a categoria disponível na tabela de produtos.

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_product_sales AS

SELECT
    i.product_id,
    p.product_category_name,

    COUNT(*) AS quantidade_itens_vendidos,
    COUNT(DISTINCT i.order_id) AS quantidade_pedidos,
    COUNT(DISTINCT i.seller_id) AS quantidade_vendedores,

    SUM(i.price) AS valor_produtos,
    SUM(i.freight_value) AS valor_frete,
    SUM(i.valor_total_item) AS valor_total_vendas

FROM workspace.default.silver_order_items i

LEFT JOIN workspace.default.silver_products p
    ON i.product_id = p.product_id

GROUP BY
    i.product_id,
    p.product_category_name;

### 9.5 Validação da tabela Gold de produtos

A validação busca confirmar que cada produto aparece uma única vez na tabela Gold e que os valores agregados foram calculados a partir dos registros da camada Silver.

In [0]:
%sql

SELECT
    COUNT(*) AS total_produtos,
    COUNT(DISTINCT product_id) AS produtos_distintos,
    COUNT(*) - COUNT(product_id) AS product_id_nulos
FROM workspace.default.gold_product_sales;

In [0]:
%sql

SELECT
    product_id,
    COUNT(*) AS quantidade
FROM workspace.default.gold_product_sales
GROUP BY product_id
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

### 9.6 Exploração inicial das vendas por produto

Como primeira exploração da camada Gold, foi realizada uma ordenação dos produtos pelo valor total das vendas, permitindo identificar os produtos com maior movimentação financeira na base.

In [0]:
%sql

SELECT
    product_id,
    product_category_name,
    quantidade_itens_vendidos,
    quantidade_pedidos,
    valor_total_vendas
FROM workspace.default.gold_product_sales
ORDER BY valor_total_vendas DESC
LIMIT 10;

### 9.7 Visão de vendas por cliente

A tabela `gold_customer_sales` consolida o comportamento de compra dos clientes a partir do identificador `customer_unique_id`.

A utilização desse identificador permite analisar o cliente ao longo de diferentes pedidos, possibilitando a identificação de clientes com compras recorrentes.

A granularidade da tabela será de um registro por cliente único.

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_customer_sales AS

SELECT
    c.customer_unique_id,

    COUNT(DISTINCT o.order_id) AS quantidade_pedidos,
    SUM(o.quantidade_itens) AS quantidade_itens,
    SUM(o.valor_produtos) AS valor_produtos,
    SUM(o.valor_frete) AS valor_frete,
    SUM(o.valor_total_itens) AS valor_total_compras,
    SUM(o.valor_total_pago) AS valor_total_pago,

    MIN(o.order_purchase_timestamp) AS primeira_compra,
    MAX(o.order_purchase_timestamp) AS ultima_compra

FROM workspace.default.silver_customers c

INNER JOIN workspace.default.gold_orders o
    ON c.customer_id = o.customer_id

GROUP BY
    c.customer_unique_id;

### 9.8 Validação da tabela Gold de clientes

A validação busca confirmar que a tabela apresenta um único registro por `customer_unique_id` e que os indicadores de compra foram agregados corretamente.

In [0]:
%sql

SELECT
    COUNT(*) AS total_clientes,
    COUNT(DISTINCT customer_unique_id) AS clientes_distintos,
    COUNT(*) - COUNT(customer_unique_id) AS customer_unique_id_nulos
FROM workspace.default.gold_customer_sales;

In [0]:
%sql

SELECT
    customer_unique_id,
    COUNT(*) AS quantidade
FROM workspace.default.gold_customer_sales
GROUP BY customer_unique_id
HAVING COUNT(*) > 1
ORDER BY quantidade DESC;

### 9.9 Análise inicial de recorrência

A partir da tabela Gold de clientes, foi realizada uma análise da quantidade de pedidos por cliente, permitindo distinguir clientes com uma única compra daqueles que realizaram compras recorrentes.

In [0]:
%sql

SELECT
    quantidade_pedidos,
    COUNT(*) AS quantidade_clientes
FROM workspace.default.gold_customer_sales
GROUP BY quantidade_pedidos
ORDER BY quantidade_pedidos;

In [0]:
%sql

SELECT
    COUNT(*) AS total_clientes,
    SUM(CASE WHEN quantidade_pedidos = 1 THEN 1 ELSE 0 END) AS clientes_uma_compra,
    SUM(CASE WHEN quantidade_pedidos > 1 THEN 1 ELSE 0 END) AS clientes_recorrentes
FROM workspace.default.gold_customer_sales;

Quase 3.000 clientes realizaram mais de uma compra, enquanto 93000 fizeram apenas uma.

### Resultado da análise de recorrência

A análise identificou 96.096 clientes únicos na base. Destes, 93.099 realizaram apenas uma compra, enquanto 2.997 realizaram mais de uma compra.

Os clientes recorrentes representam aproximadamente 3,12% da base de clientes únicos. A distribuição mostra que a maior parte dos clientes recorrentes realizou duas compras, enquanto uma parcela menor apresentou três ou mais pedidos.

Essa informação será utilizada posteriormente na análise do comportamento dos clientes.

## 10. Análise de Dados

A camada Gold disponibiliza as informações consolidadas necessárias para responder às perguntas de negócio.

Nesta etapa serão realizadas análises descritivas sobre evolução dos pedidos, movimentação financeira, produtos, comportamento dos clientes e avaliações.

Os resultados serão apresentados por meio de consultas SQL e utilizados para interpretar os principais padrões observados na base.

### 10.1 Evolução do volume de pedidos

A evolução temporal do volume de pedidos foi analisada por mês de compra, permitindo identificar períodos de maior e menor movimentação na plataforma.

In [0]:
%sql

SELECT
    ano_compra,
    mes_compra,
    COUNT(*) AS quantidade_pedidos
FROM workspace.default.gold_orders
GROUP BY
    ano_compra,
    mes_compra
ORDER BY
    ano_compra,
    mes_compra;

### 10.2 Evolução do valor das vendas

Além do volume de pedidos, foi analisada a evolução mensal do valor dos produtos vendidos, permitindo comparar a movimentação financeira ao longo do período.

In [0]:
%sql

SELECT
    ano_compra,
    mes_compra,
    COUNT(*) AS quantidade_pedidos,
    ROUND(SUM(valor_produtos), 2) AS valor_vendas,
    ROUND(SUM(valor_frete), 2) AS valor_frete
FROM workspace.default.gold_orders
GROUP BY
    ano_compra,
    mes_compra
ORDER BY
    ano_compra,
    mes_compra;

### 10.3 Evolução do ticket médio

O ticket médio foi calculado como a razão entre o valor total dos produtos vendidos e a quantidade de pedidos em cada período.

In [0]:
%sql

SELECT
    ano_compra,
    mes_compra,
    COUNT(*) AS quantidade_pedidos,
    ROUND(SUM(valor_produtos), 2) AS valor_vendas,
    ROUND(SUM(valor_produtos) / COUNT(*), 2) AS ticket_medio
FROM workspace.default.gold_orders
GROUP BY
    ano_compra,
    mes_compra
ORDER BY
    ano_compra,
    mes_compra;

### 10.4 Identificação dos períodos de maior e menor movimentação

A partir da série mensal, foram identificados os períodos com maior e menor quantidade de pedidos e maior e menor valor de vendas. Essa comparação permite complementar a análise da evolução temporal e destacar os principais pontos de concentração da movimentação.

In [0]:
%sql

SELECT
    ano_compra,
    mes_compra,
    quantidade_pedidos,
    valor_vendas,
    ticket_medio
FROM (
    SELECT
        ano_compra,
        mes_compra,
        COUNT(*) AS quantidade_pedidos,
        ROUND(SUM(valor_produtos), 2) AS valor_vendas,
        ROUND(SUM(valor_produtos) / COUNT(*), 2) AS ticket_medio
    FROM workspace.default.gold_orders
    GROUP BY
        ano_compra,
        mes_compra
)
ORDER BY quantidade_pedidos DESC
LIMIT 5;

In [0]:
%sql

SELECT
    ano_compra,
    mes_compra,
    quantidade_pedidos,
    valor_vendas,
    ticket_medio
FROM (
    SELECT
        ano_compra,
        mes_compra,
        COUNT(*) AS quantidade_pedidos,
        ROUND(SUM(valor_produtos), 2) AS valor_vendas,
        ROUND(SUM(valor_produtos) / COUNT(*), 2) AS ticket_medio
    FROM workspace.default.gold_orders
    GROUP BY
        ano_compra,
        mes_compra
)
ORDER BY valor_vendas DESC
LIMIT 5;

### 10.5 Evolução anual das vendas

Para facilitar a interpretação da tendência geral, os indicadores mensais foram consolidados por ano, permitindo comparar o volume de pedidos, o valor movimentado e o ticket médio entre os períodos.

In [0]:
%sql

SELECT
    ano_compra,
    COUNT(*) AS quantidade_pedidos,
    ROUND(SUM(valor_produtos), 2) AS valor_vendas,
    ROUND(SUM(valor_produtos) / COUNT(*), 2) AS ticket_medio
FROM workspace.default.gold_orders
GROUP BY ano_compra
ORDER BY ano_compra;

### Resultado da análise

A análise temporal evidencia uma forte expansão do volume de pedidos entre o início da série e os períodos posteriores. A base registra 329 pedidos em 2016, 45.101 em 2017 e 54.011 em 2018 no período disponível.

O valor total dos produtos vendidos acompanhou essa evolução, passando de R$ 49,8 mil em 2016 para aproximadamente R$ 6,16 milhões em 2017 e R$ 7,39 milhões em 2018.

Apesar do crescimento do volume e do valor movimentado, o ticket médio apresentou comportamento diferente. O valor médio por pedido passou de R$ 151,32 em 2016 para R$ 136,49 em 2017, permanecendo próximo desse nível em 2018, com R$ 136,75.

A interpretação dos valores anuais deve considerar que 2016 e 2018 não representam necessariamente períodos completos na base. Dessa forma, a análise indica uma expansão da movimentação no período observado, mas não deve ser interpretada como uma comparação entre três anos completos.

### 10.6 Vendas por categoria de produto

A distribuição das vendas por categoria foi analisada para identificar quais categorias concentram maior volume financeiro e quantidade de itens vendidos.

A análise utiliza a tabela `gold_product_sales`, agregando os produtos por categoria.

In [0]:
%sql

SELECT
    product_category_name,
    SUM(quantidade_itens_vendidos) AS quantidade_itens,
    SUM(quantidade_pedidos) AS quantidade_pedidos,
    ROUND(SUM(valor_total_vendas), 2) AS valor_total_vendas
FROM workspace.default.gold_product_sales
GROUP BY product_category_name
ORDER BY valor_total_vendas DESC
LIMIT 10;

In [0]:
%sql

SELECT
    product_category_name,
    SUM(quantidade_itens_vendidos) AS quantidade_itens,
    ROUND(SUM(valor_total_vendas), 2) AS valor_total_vendas
FROM workspace.default.gold_product_sales
GROUP BY product_category_name
ORDER BY quantidade_itens DESC
LIMIT 10;

### Resultado da análise por categoria

A análise das categorias mostra diferenças entre volume de itens vendidos e valor total das vendas.

Por valor total de vendas, as categorias `beleza_saude`, `relogios_presentes` e `cama_mesa_banho` apresentaram os maiores valores entre as categorias analisadas, com aproximadamente R$ 1,41 milhão, R$ 1,31 milhão e R$ 1,25 milhão, respectivamente.

Quando analisada a quantidade de itens vendidos, `cama_mesa_banho` apresentou o maior volume, com 11.115 itens, seguida por `beleza_saude`, com 9.670 itens.

A diferença entre os rankings indica que o volume de itens vendidos e o valor movimentado não são necessariamente proporcionais. 

### 10.7 Valor médio por item por categoria

Para complementar a análise, foi calculado o valor médio dos itens vendidos em cada categoria. O indicador permite comparar categorias considerando não apenas o volume de vendas, mas também o valor médio dos produtos comercializados.

In [0]:
%sql

SELECT
    product_category_name,
    SUM(quantidade_itens_vendidos) AS quantidade_itens,
    ROUND(SUM(valor_total_vendas), 2) AS valor_total_vendas,
    ROUND(
        SUM(valor_total_vendas) / SUM(quantidade_itens_vendidos),
        2
    ) AS valor_medio_item
FROM workspace.default.gold_product_sales
WHERE product_category_name IS NOT NULL
GROUP BY product_category_name
HAVING SUM(quantidade_itens_vendidos) >= 1000
ORDER BY valor_medio_item DESC
LIMIT 10;

### Resultado da análise de valor médio por categoria

A análise do valor médio por item mostra que as categorias apresentam comportamentos distintos quando considerado o valor unitário das vendas.

Entre as categorias com pelo menos 1.000 itens vendidos, `relogios_presentes` apresentou o maior valor médio por item, de R$ 217,92, seguida por `moveis_escritorio`, com R$ 202,56.

Por outro lado, `beleza_saude`, que apresentou o maior valor total de vendas entre as categorias analisadas, possui valor médio de R$ 149,04 por item e um volume significativamente maior de itens vendidos.

Os resultados reforçam que o valor total movimentado por uma categoria depende da combinação entre quantidade de itens vendidos e valor médio dos itens, não sendo possível analisar o desempenho das categorias considerando apenas uma dessas dimensões.

### 10.8 Comportamento de compra dos clientes

A análise do comportamento dos clientes foi complementada pela comparação entre clientes de compra única e clientes recorrentes.

Além da quantidade de pedidos, será analisado o valor total das compras de cada grupo.

In [0]:
%sql

SELECT
    CASE
        WHEN quantidade_pedidos = 1 THEN 'Compra única'
        ELSE 'Recorrente'
    END AS tipo_cliente,

    COUNT(*) AS quantidade_clientes,

    ROUND(SUM(valor_total_compras), 2) AS valor_total_compras,

    ROUND(
        SUM(valor_total_compras) / COUNT(*),
        2
    ) AS valor_medio_por_cliente

FROM workspace.default.gold_customer_sales

GROUP BY
    CASE
        WHEN quantidade_pedidos = 1 THEN 'Compra única'
        ELSE 'Recorrente'
    END

ORDER BY
    tipo_cliente;

### Resultado da análise de comportamento dos clientes

A análise identificou uma diferença relevante entre a quantidade de clientes e o valor movimentado por cada grupo.

Os clientes recorrentes representam 2.997 dos 96.096 clientes únicos da base, aproximadamente 3,12%. Apesar de representarem uma parcela reduzida da quantidade de clientes, esse grupo movimentou R$ 922.042,49, correspondendo a aproximadamente 38,2% do valor total de compras analisado.

O valor médio de compras por cliente recorrente foi de R$ 307,66, enquanto clientes que realizaram apenas uma compra apresentaram valor médio de R$ 160,28.

Os resultados mostram que a frequência de compra está associada a uma maior concentração de valor movimentado na base. Essa análise é descritiva e não permite, isoladamente, estabelecer uma relação causal entre recorrência e maior valor de compras.

### 10.9 Distribuição das avaliações dos clientes

A distribuição das avaliações foi analisada a partir da nota atribuída aos pedidos. O objetivo é identificar a concentração das avaliações nas diferentes faixas da escala de 1 a 5.

In [0]:
%sql

SELECT
    review_score,
    COUNT(*) AS quantidade_avaliacoes,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentual_avaliacoes
FROM workspace.default.silver_order_reviews
WHERE review_score IS NOT NULL
GROUP BY review_score
ORDER BY review_score;

### Resultado da análise das avaliações

A distribuição das avaliações apresenta concentração nas notas mais altas. A nota 5 representa 57,78% das avaliações com nota disponível, seguida pela nota 4, com 19,29%.

As notas 1 e 2 representam, respectivamente, 11,51% e 3,18% das avaliações, enquanto a nota 3 corresponde a 8,24%.

Dessa forma, 77,07% das avaliações com nota disponível apresentam pontuação 4 ou 5. A análise descreve a distribuição das avaliações presentes na base, sem buscar identificar relações causais para os resultados observados.

## 11. Síntese dos Resultados

As análises realizadas permitiram responder às principais perguntas propostas.

A análise temporal identificou expansão do volume de pedidos e do valor movimentado ao longo do período disponível na base, enquanto o ticket médio apresentou comportamento relativamente mais estável nos períodos de maior volume.

Na análise de produtos, foram observadas diferenças entre as categorias com maior quantidade de itens vendidos e aquelas com maior valor financeiro. `cama_mesa_banho` apresentou o maior volume de itens, enquanto `beleza_saude` apresentou o maior valor total de vendas. A análise do valor médio por item demonstrou ainda que categorias com menor volume podem apresentar valores unitários superiores.

Em relação aos clientes, foram identificados 96.096 clientes únicos. Destes, 2.997 realizaram mais de uma compra, correspondendo a aproximadamente 3,12% da base. Apesar da menor participação em quantidade de clientes, o grupo recorrente apresentou valor médio de compras superior ao grupo de compra única.

Por fim, a análise das avaliações identificou concentração nas notas mais altas, com 77,07% das avaliações válidas apresentando notas 4 ou 5.

Em conjunto, os resultados demonstram como a construção do pipeline e das camadas Bronze, Silver e Gold permite transformar os dados brutos em informações estruturadas para análise do comportamento de vendas, produtos e clientes.

## 12. Qualidade dos Dados

A qualidade dos dados foi avaliada ao longo da construção do pipeline, considerando principalmente completude, unicidade, consistência e validade dos atributos.

Entre as verificações realizadas estão:

- identificação de valores nulos em campos utilizados nos relacionamentos;
- verificação de duplicidades nos identificadores das principais entidades;
- validação da granularidade das tabelas;
- identificação de valores negativos em atributos financeiros e físicos;
- validação dos valores possíveis das avaliações;
- análise da relação entre `customer_id` e `customer_unique_id`;
- verificação da cardinalidade entre pedidos, itens e pagamentos.

Foram identificados alguns problemas de completude, como produtos sem categoria e avaliações com informações ausentes. Esses registros foram preservados quando sua utilização analítica continuava sendo possível.

Na tabela de avaliações, registros sem identificação do pedido foram mantidos na camada Bronze, mas não utilizados na camada Silver, pois não poderiam ser relacionados às demais entidades.

A separação entre Bronze e Silver permitiu preservar os dados provenientes da fonte enquanto a camada tratada disponibilizou informações adequadas para a construção das estruturas analíticas da camada Gold.

## 13. Visão Geral do Pipeline

O pipeline desenvolvido neste MVP segue uma arquitetura em camadas inspirada no modelo Medallion.

**Bronze:** armazenamento dos dados provenientes dos arquivos CSV, preservando as informações da fonte e permitindo a identificação de problemas de qualidade.

**Silver:** tratamento e validação dos dados, incluindo definição de tipos, análise de valores nulos, validação de identificadores e preservação da granularidade das entidades.

**Gold:** construção de estruturas analíticas consolidadas para pedidos, produtos e clientes, utilizadas para responder às perguntas de negócio.

O fluxo implementado pode ser resumido como:

`Arquivos CSV → Volume Databricks → Bronze → Silver → Gold → Análises`

A utilização das diferentes camadas permitiu separar as responsabilidades de ingestão, tratamento e consumo analítico dos dados.

## 14. Autoavaliação

O MVP atingiu o objetivo de construir um pipeline de dados em ambiente de nuvem, partindo de arquivos CSV e chegando a estruturas analíticas capazes de responder às perguntas de negócio propostas.

O desenvolvimento permitiu aplicar conceitos de ingestão, modelagem, transformação, qualidade e análise de dados utilizando Databricks e SQL.

Um dos principais aprendizados foi a importância da granularidade e da cardinalidade durante a integração das tabelas. As relações de um para muitos entre pedidos, itens e pagamentos exigiram agregações prévias para evitar a multiplicação de registros e a distorção dos indicadores financeiros.

Também foi possível observar na prática a função das diferentes camadas da arquitetura Medallion e a importância de preservar os dados originais antes da aplicação dos tratamentos.

Como evolução futura, o projeto poderia incorporar novas fontes de dados, automatizar etapas adicionais do pipeline e ampliar as análises e visualizações produzidas a partir da camada Gold.